# Import necessary dependencies and settings

In [1]:
import datetime
import numpy as np
import pandas as pd
from dateutil.parser import parse
import pytz

# Load and process sample temporal data

In [2]:
time_stamps = ['2015-03-08 10:30:00.360000+00:00', '2017-07-13 15:45:05.755000-07:00',
               '2012-01-20 22:30:00.254000+05:30', '2016-12-25 00:30:00.000000+10:00']
df = pd.DataFrame(time_stamps, columns=['Time'])
df

,Time
0,2015-03-08 10:30:00.360000+00:00
1,2017-07-13 15:45:05.755000-07:00
2,2012-01-20 22:30:00.254000+05:30
3,2016-12-25 00:30:00.000000+10:00


In [3]:
ts_objs = np.array([pd.Timestamp(item) for item in np.array(df.Time)])
df['TS_obj'] = ts_objs
ts_objs

array([Timestamp('2015-03-08 10:30:00.360000+0000', tz='UTC'),
       Timestamp('2017-07-13 15:45:05.755000-0700', tz='UTC-07:00'),
       Timestamp('2012-01-20 22:30:00.254000+0530', tz='UTC+05:30'),
       Timestamp('2016-12-25 00:30:00+1000', tz='UTC+10:00')],
      dtype=object)

# Date based features

In [4]:
df['Year'] = df['TS_obj'].apply(lambda d: d.year)
df['Month'] = df['TS_obj'].apply(lambda d: d.month)
df['Day'] = df['TS_obj'].apply(lambda d: d.day)
df['DayOfWeek'] = df['TS_obj'].apply(lambda d: d.dayofweek)
df['DayName'] = df['TS_obj'].apply(lambda d: d.weekday_name)
df['DayOfYear'] = df['TS_obj'].apply(lambda d: d.dayofyear)
df['WeekOfYear'] = df['TS_obj'].apply(lambda d: d.weekofyear)
df['Quarter'] = df['TS_obj'].apply(lambda d: d.quarter)

df[['Time', 'Year', 'Month', 'Day', 'Quarter',
    'DayOfWeek', 'DayName', 'DayOfYear', 'WeekOfYear']]

AttributeError: 'Timestamp' object has no attribute 'weekday_name'

# Time based features

In [ ]:
df['Hour'] = df['TS_obj'].apply(lambda d: d.hour)
df['Minute'] = df['TS_obj'].apply(lambda d: d.minute)
df['Second'] = df['TS_obj'].apply(lambda d: d.second)
df['MUsecond'] = df['TS_obj'].apply(lambda d: d.microsecond)
df['UTC_offset'] = df['TS_obj'].apply(lambda d: d.utcoffset())

df[['Time', 'Hour', 'Minute', 'Second', 'MUsecond', 'UTC_offset']]

In [ ]:
hour_bins = [-1, 5, 11, 16, 21, 23]
bin_names = ['Late Night', 'Morning', 'Afternoon', 'Evening', 'Night']
df['TimeOfDayBin'] = pd.cut(df['Hour'],
                            bins=hour_bins, labels=bin_names)
df[['Time', 'Hour', 'TimeOfDayBin']]

In [ ]:
df['TZ_info'] = df['TS_obj'].apply(lambda d: d.tzinfo)
df['TimeZones'] = df['TS_obj'].apply(lambda d: list({d.astimezone(tz).tzname()
                                   for tz in map(pytz.timezone,
                                                 pytz.all_timezones_set)
                                       if d.astimezone(tz).utcoffset() == d.utcoffset()}))

df[['Time', 'UTC_offset', 'TZ_info', 'TimeZones']]

In [ ]:
df['TimeUTC'] = df['TS_obj'].apply(lambda d: d.tz_convert(pytz.utc))
df['Epoch'] = df['TimeUTC'].apply(lambda d: d.timestamp())
df['GregOrdinal'] = df['TimeUTC'].apply(lambda d: d.toordinal())

df[['Time', 'TimeUTC', 'Epoch', 'GregOrdinal']]

In [ ]:
curr_ts = datetime.datetime.now(pytz.utc)
# compute days elapsed since today
df['DaysElapsedEpoch'] = (curr_ts.timestamp() - df['Epoch']) / (3600*24)
df['DaysElapsedOrdinal'] = (curr_ts.toordinal() - df['GregOrdinal'])

df[['Time', 'TimeUTC', 'DaysElapsedEpoch', 'DaysElapsedOrdinal']]

## Binarization of `song_views.csv`

In [6]:
df_songs = pd.read_csv('/content/song_views.csv')
display(df_songs.head())
display(df_songs.info())

,user_id,song_id,title,listen_count
0,b6b799f34a204bd928ea014c243ddad6d0be4f8f,SOBONKR12A58A7A7E0,You're The One,2
1,b41ead730ac14f6b6717b9cf8859d5579f3f8d4d,SOBONKR12A58A7A7E0,You're The One,0
2,4c84359a164b161496d05282707cecbd50adbfc4,SOBONKR12A58A7A7E0,You're The One,0
3,779b5908593756abb6ff7586177c966022668b06,SOBONKR12A58A7A7E0,You're The One,0
4,dd88ea94f605a63d9fc37a214127e3f00e85e42d,SOBONKR12A58A7A7E0,You're The One,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   user_id       365 non-null    object
 1   song_id       365 non-null    object
 2   title         365 non-null    object
 3   listen_count  365 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 11.5+ KB


None

In [7]:
# Binarize 'listen_count': 1 if listened (count > 0), 0 otherwise
df_songs['has_listened'] = (df_songs['listen_count'] > 0).astype(int)

display(df_songs[['user_id', 'song_id', 'listen_count', 'has_listened']].head())

,user_id,song_id,listen_count,has_listened
0,b6b799f34a204bd928ea014c243ddad6d0be4f8f,SOBONKR12A58A7A7E0,2,1
1,b41ead730ac14f6b6717b9cf8859d5579f3f8d4d,SOBONKR12A58A7A7E0,0,0
2,4c84359a164b161496d05282707cecbd50adbfc4,SOBONKR12A58A7A7E0,0,0
3,779b5908593756abb6ff7586177c966022668b06,SOBONKR12A58A7A7E0,0,0
4,dd88ea94f605a63d9fc37a214127e3f00e85e42d,SOBONKR12A58A7A7E0,0,0


## Feature Engineering on `Pokemon.csv`

In [8]:
df_pokemon = pd.read_csv('/content/Pokemon.csv')
display(df_pokemon.head())
display(df_pokemon.info())

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,Gen 1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,Gen 1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,Gen 1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,Gen 1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,Gen 1,False


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   #           800 non-null    int64 
 1   Name        800 non-null    object
 2   Type 1      800 non-null    object
 3   Type 2      414 non-null    object
 4   Total       800 non-null    int64 
 5   HP          800 non-null    int64 
 6   Attack      800 non-null    int64 
 7   Defense     800 non-null    int64 
 8   Sp. Atk     800 non-null    int64 
 9   Sp. Def     800 non-null    int64 
 10  Speed       800 non-null    int64 
 11  Generation  800 non-null    object
 12  Legendary   800 non-null    bool  
dtypes: bool(1), int64(8), object(4)
memory usage: 75.9+ KB


None

### 1. Interaction Features

In [9]:
# Create an interaction feature: Attack_Defense_Product
df_pokemon['Attack_Defense_Product'] = df_pokemon['Attack'] * df_pokemon['Defense']

display(df_pokemon[['Name', 'Attack', 'Defense', 'Attack_Defense_Product']].head())

,Name,Attack,Defense,Attack_Defense_Product
0,Bulbasaur,49,49,2401
1,Ivysaur,62,63,3906
2,Venusaur,82,83,6806
3,VenusaurMega Venusaur,100,123,12300
4,Charmander,52,43,2236


### 2. Ordinal Feature Transformation

In [13]:
# Transform 'Generation' into a descriptive ordinal feature
gen_map = {1: 'Generation 1', 2: 'Generation 2', 3: 'Generation 3',
           4: 'Generation 4', 5: 'Generation 5', 6: 'Generation 6'}

# Extract the numeric part from 'Generation' and convert to int
df_pokemon['Generation_Numeric'] = df_pokemon['Generation'].str.extract('(\d+)').astype(int)

df_pokemon['Generation_Ordinal'] = df_pokemon['Generation_Numeric'].map(gen_map)

display(df_pokemon[['Name', 'Generation', 'Generation_Numeric', 'Generation_Ordinal']].head())
display(df_pokemon['Generation_Ordinal'].value_counts())

<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_8103/3316190079.py:6: SyntaxWarning: invalid escape sequence '\d'
  df_pokemon['Generation_Numeric'] = df_pokemon['Generation'].str.extract('(\d+)').astype(int)


,Name,Generation,Generation_Numeric,Generation_Ordinal
0,Bulbasaur,Gen 1,1,Generation 1
1,Ivysaur,Gen 1,1,Generation 1
2,Venusaur,Gen 1,1,Generation 1
3,VenusaurMega Venusaur,Gen 1,1,Generation 1
4,Charmander,Gen 1,1,Generation 1


,count
Generation_Ordinal,
Generation 1,166
Generation 5,165
Generation 3,160
Generation 4,121
Generation 2,106
Generation 6,82


### 3. One-Hot Encoding

In [12]:
# Apply one-hot encoding to 'Type 1'
df_pokemon_one_hot = pd.get_dummies(df_pokemon, columns=['Type 1'], prefix='Type1')

# Get all one-hot encoded columns
one_hot_columns = [col for col in df_pokemon_one_hot.columns if col.startswith('Type1_')]

# Display the 'Name' column along with the generated one-hot encoded columns
display(df_pokemon_one_hot[['Name'] + one_hot_columns].head())

,Name,Type1_Bug,Type1_Dark,Type1_Dragon,Type1_Electric,Type1_Fairy,Type1_Fighting,Type1_Fire,Type1_Flying,Type1_Ghost,Type1_Grass,Type1_Ground,Type1_Ice,Type1_Normal,Type1_Poison,Type1_Psychic,Type1_Rock,Type1_Steel,Type1_Water
0,Bulbasaur,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
1,Ivysaur,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
2,Venusaur,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
3,VenusaurMega Venusaur,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
4,Charmander,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False


## Scaling Numerical Features on `song_views.csv` and `item_popularity.csv`

In [14]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Load item_popularity.csv
df_popularity = pd.read_csv('/content/item_popularity.csv')
display(df_popularity.head())
display(df_popularity.info())

,item_id,pop_percent
0,it_01345,0.98324
1,it_03431,0.56123
2,it_04572,0.12098
3,it_98021,0.35476
4,it_01298,0.92101


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   item_id      7 non-null      object 
 1   pop_percent  7 non-null      float64
dtypes: float64(1), object(1)
memory usage: 244.0+ bytes


None

### Identify Numerical Columns

In [15]:
# For df_songs, 'listen_count' is the numerical column
songs_numerical_cols = ['listen_count']

# For df_popularity, identify numerical columns dynamically
popularity_numerical_cols = df_popularity.select_dtypes(include=np.number).columns.tolist()

print(f"Numerical columns in df_songs: {songs_numerical_cols}")
print(f"Numerical columns in df_popularity: {popularity_numerical_cols}")

Numerical columns in df_songs: ['listen_count']
Numerical columns in df_popularity: ['pop_percent']


### 1. Standard Scaling

In [16]:
scaler = StandardScaler()

# Standard Scaling for df_songs
df_songs_scaled_standard = df_songs.copy()
df_songs_scaled_standard[songs_numerical_cols] = scaler.fit_transform(df_songs[songs_numerical_cols])
print("Standard Scaled df_songs (first 5 rows of relevant columns):")
display(df_songs_scaled_standard[['listen_count', 'has_listened']].head())

# Standard Scaling for df_popularity
df_popularity_scaled_standard = df_popularity.copy()
df_popularity_scaled_standard[popularity_numerical_cols] = scaler.fit_transform(df_popularity[popularity_numerical_cols])
print("\nStandard Scaled df_popularity (first 5 rows of numerical columns):")
display(df_popularity_scaled_standard[popularity_numerical_cols].head())

Standard Scaled df_songs (first 5 rows of relevant columns):


,listen_count,has_listened
0,0.061911,1
1,-0.152283,0
2,-0.152283,0
3,-0.152283,0
4,-0.152283,0



Standard Scaled df_popularity (first 5 rows of numerical columns):


,pop_percent
0,1.266937
1,-0.192561
2,-1.715141
3,-0.906626
4,1.051718


### 2. Min-Max Scaling

In [17]:
scaler = MinMaxScaler()

# Min-Max Scaling for df_songs
df_songs_scaled_minmax = df_songs.copy()
df_songs_scaled_minmax[songs_numerical_cols] = scaler.fit_transform(df_songs[songs_numerical_cols])
print("Min-Max Scaled df_songs (first 5 rows of relevant columns):")
display(df_songs_scaled_minmax[['listen_count', 'has_listened']].head())

# Min-Max Scaling for df_popularity
df_popularity_scaled_minmax = df_popularity.copy()
df_popularity_scaled_minmax[popularity_numerical_cols] = scaler.fit_transform(df_popularity[popularity_numerical_cols])
print("\nMin-Max Scaled df_popularity (first 5 rows of numerical columns):")
display(df_popularity_scaled_minmax[popularity_numerical_cols].head())

Min-Max Scaled df_songs (first 5 rows of relevant columns):


,listen_count,has_listened
0,0.01626,1
1,0.00000,0
2,0.00000,0
3,0.00000,0
4,0.00000,0



Min-Max Scaled df_popularity (first 5 rows of numerical columns):


,pop_percent
0,1.000000
1,0.510577
2,0.000000
3,0.271125
4,0.927829


## Binning and Log Transformation on `fcc_2016_coder_survey_subset.csv`

In [18]:
# Load the dataset
df_fcc = pd.read_csv('/content/fcc_2016_coder_survey_subset.csv')

display(df_fcc.head())
display(df_fcc.info())

,ID.x,Age,ChildrenNumber,CountryCitizen,EmploymentField,EmploymentStatus,ExpectedEarning,Gender,Income,JobPref,JobRoleInterest,LanguageAtHome,MaritalStatus,SchoolDegree,StudentDebtOwe
0,cef35615d61b202f1dc794ef2746df14,28.0,NaN,United States of America,office and administrative support,Employed for wages,NaN,male,32000.0,freelance,NaN,English,married or domestic partnership,"some college credit, no degree",20000.0
1,323e5a113644d18185c743c241407754,22.0,NaN,United States of America,food and beverage,Employed for wages,50000.0,male,15000.0,work for a startup,Front-End Web Developer,English,NaN,"some college credit, no degree",NaN
2,b29a1027e5cd062e654a63764157461d,19.0,NaN,United States of America,finance,Employed for wages,NaN,male,48000.0,start your own business,NaN,English,NaN,high school diploma or equivalent (GED),NaN
3,04a11e4bcb573a1261eb0d9948d32637,26.0,NaN,United States of America,"arts, entertainment, sports, or media",Employed for wages,65000.0,female,43000.0,work for a startup,Front-End Web Developer,English,NaN,bachelor's degree,7000.0
4,9368291c93d5d5f5c8cdb1a575e18bec,20.0,NaN,United States of America,education,Employed for wages,45000.0,female,6000.0,work for a medium-sized company,Full-Stack Web Developer,Khmer,NaN,"some college credit, no degree",NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15620 entries, 0 to 15619
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID.x              15620 non-null  object 
 1   Age               13613 non-null  float64
 2   ChildrenNumber    2554 non-null   float64
 3   CountryCitizen    13654 non-null  object 
 4   EmploymentField   7608 non-null   object 
 5   EmploymentStatus  12904 non-null  object 
 6   ExpectedEarning   6077 non-null   float64
 7   Gender            13746 non-null  object 
 8   Income            7329 non-null   float64
 9   JobPref           11189 non-null  object 
 10  JobRoleInterest   6503 non-null   object 
 11  LanguageAtHome    13582 non-null  object 
 12  MaritalStatus     4010 non-null   object 
 13  SchoolDegree      13679 non-null  object 
 14  StudentDebtOwe    3514 non-null   float64
dtypes: float64(5), object(10)
memory usage: 1.8+ MB


None

### Identify Numerical Columns for Transformation

In [19]:
# Identify numerical columns, excluding IDs
numerical_cols_fcc = df_fcc.select_dtypes(include=np.number).columns.tolist()
# Exclude 'ID.x' and 'ID.y' if they exist and are not meant for transformation
numerical_cols_fcc = [col for col in numerical_cols_fcc if 'ID' not in col]

print(f"Numerical columns in df_fcc: {numerical_cols_fcc}")

# Let's focus on 'Age' and 'Income' for transformations as they are common candidates
# Check if 'Age' and 'Income' exist and are numeric
if 'Age' in numerical_cols_fcc:
    print("\n'Age' column is available for transformations.")
else:
    print("\n'Age' column not found or is not numerical.")

if 'Income' in numerical_cols_fcc:
    print("'Income' column is available for transformations.")
else:
    print("'Income' column not found or is not numerical.")


Numerical columns in df_fcc: ['Age', 'ChildrenNumber', 'ExpectedEarning', 'Income', 'StudentDebtOwe']

'Age' column is available for transformations.
'Income' column is available for transformations.


### 1. Fixed Binning on 'Age'

In [23]:
if 'Age' in df_fcc.columns:
    # Define fixed bins for Age
    age_bins = [0, 18, 25, 35, 50, 100] # Example bins: <18, 18-24, 25-34, 35-49, 50+
    age_bin_labels = ['<18', '18-24', '25-34', '35-49', '50+']

    df_fcc['Age_Fixed_Bins'] = pd.cut(df_fcc['Age'], bins=age_bins, labels=age_bin_labels, right=False)

    print("Fixed Binning on 'Age':")
    display(df_fcc[['Age', 'Age_Fixed_Bins']].head())
    display(df_fcc['Age_Fixed_Bins'].value_counts().sort_index())
else:
    print("'Age' column not found in the dataset. Skipping fixed binning.")

Fixed Binning on 'Age':


,Age,Age_Fixed_Bins
0,28.0,25-34
1,22.0,18-24
2,19.0,18-24
3,26.0,25-34
4,20.0,18-24


,count
Age_Fixed_Bins,
<18,588
18-24,3993
25-34,6041
35-49,2462
50+,529


### 2. Adaptive Binning (Quantile Binning) on 'Income'

In [21]:
if 'Income' in df_fcc.columns:
    # Adaptive Binning using quantiles for Income
    # Using 4 quantiles to create 4 bins (quartiles)
    df_fcc['Income_Quantile_Bins'] = pd.qcut(df_fcc['Income'], q=4, labels=False, duplicates='drop')
    # labels=False will return integer indicators for bins. You can also specify custom labels.

    print("Adaptive Binning on 'Income':")
    display(df_fcc[['Income', 'Income_Quantile_Bins']].head())
    display(df_fcc['Income_Quantile_Bins'].value_counts().sort_index())
    print("\nIncome ranges for each quantile bin:")
    display(pd.qcut(df_fcc['Income'], q=4, duplicates='drop').value_counts().sort_index())
else:
    print("'Income' column not found in the dataset. Skipping adaptive binning.")

Adaptive Binning on 'Income':


,Income,Income_Quantile_Bins
0,32000.0,1.0
1,15000.0,0.0
2,48000.0,2.0
3,43000.0,2.0
4,6000.0,0.0


,count
Income_Quantile_Bins,
0.0,2008
1.0,1676
2.0,2116
3.0,1529



Income ranges for each quantile bin:


,count
Income,
"(5999.999, 20000.0]",2008
"(20000.0, 37000.0]",1676
"(37000.0, 60000.0]",2116
"(60000.0, 200000.0]",1529


### 3. Log Transformation on 'Income'

In [22]:
if 'Income' in df_fcc.columns:
    # Apply log transformation to 'Income'
    # Using np.log1p (log(1+x)) to handle potential zero values gracefully
    df_fcc['Income_Log_Transformed'] = np.log1p(df_fcc['Income'])

    print("Log Transformation on 'Income':")
    display(df_fcc[['Income', 'Income_Log_Transformed']].head())
else:
    print("'Income' column not found in the dataset. Skipping log transformation.")

Log Transformation on 'Income':


,Income,Income_Log_Transformed
0,32000.0,10.373522
1,15000.0,9.615872
2,48000.0,10.778977
3,43000.0,10.668979
4,6000.0,8.699681
